### **Top 10 Anomalous Positive Products (with price + quantity)**

In [0]:
%sql
WITH product_stats AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue,
        AVG(Price) AS avg_price,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Description
)

SELECT
    e.Description,
    p.total_revenue,
    p.avg_price,
    p.total_qty
FROM elasticity_table e
JOIN product_stats p
ON e.Description = p.Description
WHERE e.category = 'anomalous_positive'
ORDER BY p.total_revenue DESC
LIMIT 10

Description,total_revenue,avg_price,total_qty
JUMBO BAG RED RETROSPOT,150935.55999999988,2.3859215570627805,79279
CHILLI LIGHTS,85489.9100000002,6.266393581081053,16840
JUMBO BAG STRAWBERRY,69565.8900000007,2.333300209205,37337
HOT WATER BOTTLE TEA AND SYMPATHY,63517.75,4.775815602836817,12462
JUMBO STORAGE BAG SUKI,62285.28000000077,2.5313594662218306,29503
JUMBO BAG BAROQUE BLACK WHITE,60377.490000000595,2.420743801652884,31266
JUMBO SHOPPER VINTAGE RED PAISLEY,58275.45000000102,2.442576956904124,27732
JUMBO BAG PINK POLKADOT,50847.80000000041,2.5648834110592875,25627
STRAWBERRY CERAMIC TRINKET BOX,48626.759999999806,1.5184548825710789,37610
LUNCH BAG BLACK SKULL.,46422.49000000018,2.0195611916264005,27126


### **Top 10 Inelastic Products (with price + quantity)**

In [0]:
%sql
WITH product_stats AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue,
        AVG(Price) AS avg_price,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Description
)

SELECT
    e.Description,
    p.total_revenue,
    p.avg_price,
    p.total_qty
FROM elasticity_table e
JOIN product_stats p
ON e.Description = p.Description
WHERE e.category = 'inelastic'
ORDER BY p.total_revenue DESC
LIMIT 10

Description,total_revenue,avg_price,total_qty
REGENCY CAKESTAND 3 TIER,344563.2500000036,14.175267175572394,27577
Manual,341104.90000000066,379.82535147392326,10056
DOTCOM POSTAGE,322657.48000000016,224.69183844011152,1436
WHITE HANGING HEART T-LIGHT HOLDER,266923.5500000072,3.134882312218979,96683
PARTY BUNTING,149187.04999999984,5.72343430656962,28378
ASSORTED COLOUR BIRD ORNAMENT,132187.9200000002,1.855735194009458,81809
POSTAGE,127597.42,29.29030752916225,5461
PAPER CHAIN KIT 50'S CHRISTMAS,123141.53999999803,3.358867151956241,36581
MEDIUM CERAMIC TOP STORAGE JAR,81700.92000000007,1.4684799999999996,78033
ROTATING SILVER ANGELS T-LIGHT HLDR,74448.92000000032,3.337571770334949,32525


### **Top 10 Elastic Products (with price + quantity)**

In [0]:
%sql
WITH product_stats AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue,
        AVG(Price) AS avg_price,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Description
)

SELECT
    e.Description,
    p.total_revenue,
    p.avg_price,
    p.total_qty
FROM elasticity_table e
JOIN product_stats p
ON e.Description = p.Description
WHERE e.category = 'elastic'
ORDER BY p.total_revenue DESC
LIMIT 10

Description,total_revenue,avg_price,total_qty
RABBIT NIGHT LIGHT,66964.99000000028,2.3805019305019,30788
VINTAGE UNION JACK MEMOBOARD,52122.729999999916,10.230072332730476,7200
HAND WARMER SCOTTY DOG DESIGN,30426.54000000007,2.314727999999959,13772
DOOR MAT UNION FLAG,24808.259999999904,7.381030927834966,3780
RED RETROSPOT MINI CASES,15782.90999999996,8.236254681647868,2151
DOOR MAT NEW ENGLAND,13224.079999999902,7.542618025750993,2026
CANDLEHOLDER PINK HANGING HEART,12797.120000000037,3.13683881064165,4513
SOMBRERO,12456.700000000004,1.799230769230766,9570
ANTIQUE LILY FAIRY LIGHTS,12239.510000000006,4.980875420875444,2686
"""ASSORTED FLOWER COLOUR """"LEIS""""""",11966.040000000017,0.8417142857142884,24801


### **Top 15 products that contributed highest revenue overall**

In [0]:
%sql
WITH revenue AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue
    FROM retail
    GROUP BY Description
),

joined AS (
    SELECT
        e.Description,
        e.elasticity,
        e.category,
        r.total_revenue
    FROM elasticity_table e
    JOIN revenue r
    ON e.Description = r.Description
),

ranked AS (
    SELECT
        *,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
        SUM(total_revenue) OVER () AS total_revenue_all,
        SUM(total_revenue) OVER (ORDER BY total_revenue DESC 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue
    FROM joined
)

SELECT
    Description,
    elasticity,
    category,
    total_revenue,
    revenue_rank,
    cumulative_revenue,
    (cumulative_revenue / total_revenue_all) * 100 AS cumulative_pct
FROM ranked
ORDER BY revenue_rank
limit (15)


Description,elasticity,category,total_revenue,revenue_rank,cumulative_revenue,cumulative_pct
REGENCY CAKESTAND 3 TIER,0.0,inelastic,344563.2500000036,1,344563.2500000036,1.680130172226913
Manual,0.0,inelastic,341104.90000000066,2,685668.1500000043,3.343397030733847
DOTCOM POSTAGE,0.0,inelastic,322657.48000000016,3,1008325.6300000045,4.916712140347819
WHITE HANGING HEART T-LIGHT HOLDER,-0.775,inelastic,266923.5500000072,4,1275249.1800000118,6.218262175161242
JUMBO BAG RED RETROSPOT,0.582,anomalous_positive,150935.55999999988,5,1426184.7400000116,6.95424138483599
PARTY BUNTING,-0.49,inelastic,149187.04999999984,6,1575371.7900000114,7.6816946579593495
ASSORTED COLOUR BIRD ORNAMENT,-0.116,inelastic,132187.9200000002,7,1707559.7100000116,8.326258211373462
POSTAGE,0.0,inelastic,127597.42,8,1835157.1300000115,8.948437957008865
PAPER CHAIN KIT 50'S CHRISTMAS,-0.029,inelastic,123141.53999999803,9,1958298.6700000095,9.548890317521721
CHILLI LIGHTS,0.838,anomalous_positive,85489.9100000002,10,2043788.5800000096,9.965748984869334


## **The Pricing strategy table**

### Below is the final business aggregate table that marks goods with different elasticity threshold to be categorized as whether to INCREASE VOLUME or PRICE or weather they are LUXUARY goods with always PREMIUM PRICING and lastly goods that are to be TESTED.

### The table includes high revenue products first along with their elasticity value.

In [0]:
%sql
WITH product_stats AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue,
        AVG(Price) AS avg_price,
        SUM(Quantity) AS total_qty
    FROM retail
    GROUP BY Description
),

thresholds AS (
    SELECT
        percentile_approx(total_revenue, 0.75) AS rev_p75,
        percentile_approx(total_qty, 0.75) AS qty_p75
    FROM product_stats
)

SELECT
    e.Description,
    e.category,
    e.elasticity,
    p.total_revenue,
    p.avg_price,
    p.total_qty,

    CASE
        -- Strong inelastic + high demand → safe price increase
        WHEN e.category = 'inelastic'
             AND ABS(e.elasticity) < 0.5
             AND p.total_qty >= t.qty_p75
        THEN 'Increase Price (Safe)'

        -- Elastic + good demand → volume strategy
        WHEN e.category = 'elastic'
             AND ABS(e.elasticity) > 1
             AND p.total_qty >= t.qty_p75
        THEN 'Increase Volume'

        -- Anomalous + high revenue → premium pricing
        WHEN e.category = 'anomalous_positive'
             AND p.total_revenue >= t.rev_p75
        THEN 'Premium Pricing'

        -- Others → uncertain
        ELSE 'Monitor / Test'
    END AS pricing_strategy

FROM elasticity_table e
JOIN product_stats p
ON e.Description = p.Description
CROSS JOIN thresholds t
ORDER BY p.total_revenue DESC

Description,category,elasticity,total_revenue,avg_price,total_qty,pricing_strategy
REGENCY CAKESTAND 3 TIER,inelastic,0.0,344563.2500000036,14.175267175572394,27577,Increase Price (Safe)
Manual,inelastic,0.0,341104.90000000066,379.82535147392326,10056,Increase Price (Safe)
DOTCOM POSTAGE,inelastic,0.0,322657.48000000016,224.69183844011152,1436,Monitor / Test
WHITE HANGING HEART T-LIGHT HOLDER,inelastic,-0.775,266923.5500000072,3.134882312218979,96683,Monitor / Test
JUMBO BAG RED RETROSPOT,anomalous_positive,0.582,150935.55999999988,2.3859215570627805,79279,Premium Pricing
PARTY BUNTING,inelastic,-0.49,149187.04999999984,5.72343430656962,28378,Increase Price (Safe)
ASSORTED COLOUR BIRD ORNAMENT,inelastic,-0.116,132187.9200000002,1.855735194009458,81809,Increase Price (Safe)
POSTAGE,inelastic,0.0,127597.42,29.29030752916225,5461,Increase Price (Safe)
PAPER CHAIN KIT 50'S CHRISTMAS,inelastic,-0.029,123141.53999999803,3.358867151956241,36581,Increase Price (Safe)
CHILLI LIGHTS,anomalous_positive,0.838,85489.9100000002,6.266393581081053,16840,Premium Pricing
